# 05_edges — Co-occurrence edges + SVO

> **Environment:** requires the project venv **`.venv311`** (Python 3.11) as the Jupyter kernel — the pipeline dependencies are installed only there. `run_all.command` uses it automatically. See `README.md` → Environment setup.

**Input:** `data/interim/sentences_tagged.jsonl`  
**Output:** `data/output/edges/edges_{window}.jsonl` (one per window), and optionally `data/output/edges/svo_triples.jsonl`

Per README §6: for every sentence with at least one actor and one concept, emit (actor, concept) edges, aggregate by window, normalize by per-window article count, log source contributions.

SVO extraction runs as a separate parallel track on the same sentence set. It's exploratory — used in notebook 07 to interpret strong co-occurrence edges, not to replace them.

## Pipeline steps in this notebook

1. Setup & paths
2. Load tagged sentences
3. Per-window article counts (denominator for normalization)
4. Aggregate co-occurrence edges + source breakdown
5. Quality report (top edges, dyads per window)
6. Write `edges_{window}.jsonl` files
7. SVO extraction with CAMEO verb filter — re-parses bodies, writes `svo_triples.jsonl`

## Step 1: Setup & paths

In [ ]:
import json
import sys
from pathlib import Path
from collections import defaultdict, Counter

_cwd = Path().resolve()
ROOT = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / 'src').is_dir()),
    _cwd,
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

INTERIM_DIR = ROOT / 'data' / 'interim'
EDGES_DIR   = ROOT / 'data' / 'output' / 'edges'
EDGES_DIR.mkdir(parents=True, exist_ok=True)

IN_FILE  = INTERIM_DIR / 'sentences_tagged.jsonl'

print(f'Input     : {IN_FILE}')
print(f'Edges dir : {EDGES_DIR}')
assert IN_FILE.exists(), f'ERROR: {IN_FILE} not found — run 04_extract first'

## Step 2: Load tagged sentences

In [ ]:
with open(IN_FILE, encoding='utf-8') as f:
    sentences = [json.loads(l) for l in f]

print(f'Loaded {len(sentences)} tagged sentences')

# Only sentences with at least one actor AND one concept produce edges
edge_sentences = [s for s in sentences if s['actors'] and s['concepts']]
print(f'Edge-bearing sentences: {len(edge_sentences)}  '
      f'({100*len(edge_sentences)/max(len(sentences),1):.1f}%)')

## Step 3: Per-window article counts

Normalization denominator: **number of distinct articles per window**, not sentences. 
This makes weights comparable across windows of different sizes — a 30-article month and a 6-day climax window with 200 articles each contribute fairly to the same dyad.

In [ ]:
articles_per_window = defaultdict(set)
for s in sentences:
    articles_per_window[s['window']].add(s['article_id'])
n_articles_per_window = {w: len(ids) for w, ids in articles_per_window.items()}

print('Articles per window:')
for w in sorted(n_articles_per_window):
    print(f'  {w:20s}  {n_articles_per_window[w]} articles')

## Step 4: Aggregate co-occurrence edges

Logic per README §6: for each edge-bearing sentence, every (actor × concept) pair is a co-occurrence event. 
Counts are aggregated by `(window, actor, concept)`. Source attribution is tracked so notebook 07 can do US-vs-non-Western source comparisons on individual edges.

In [ ]:
# key = (window, actor, concept)
raw_edges       = defaultdict(int)
source_breakdown = defaultdict(lambda: defaultdict(int))  # key -> source -> count

for s in edge_sentences:
    for actor in s['actors']:
        for concept in s['concepts']:
            key = (s['window'], actor, concept)
            raw_edges[key] += 1
            source_breakdown[key][s['source'] or 'UNKNOWN'] += 1

print(f'Total (window, actor, concept) keys: {len(raw_edges)}')
print(f'Total co-occurrence events         : {sum(raw_edges.values())}')
windows_present = sorted({k[0] for k in raw_edges})
print(f'Windows with at least one edge     : {windows_present}')

## Step 5: Quality report

In [ ]:
for window in windows_present:
    window_edges = {k: v for k, v in raw_edges.items() if k[0] == window}
    n_arts = n_articles_per_window.get(window, 1)
    print(f'\n========== {window}  ({n_arts} articles, {len(window_edges)} distinct dyads) ==========')
    print(f'{"weight":>7}  {"w_norm":>8}  actor — concept')
    ranked = sorted(window_edges.items(), key=lambda kv: -kv[1])
    for (w, actor, concept), wt in ranked[:15]:
        w_norm = wt / n_arts
        print(f'  {wt:5d}  {w_norm:8.3f}  {actor:18s} — {concept}')
    if len(ranked) > 15:
        print(f'  ... ({len(ranked) - 15} more dyads)')

## Step 6: Write edges_{window}.jsonl

In [ ]:
for window in windows_present:
    n_arts = n_articles_per_window[window]
    out_path = EDGES_DIR / f'edges_{window}.jsonl'
    n_written = 0
    with open(out_path, 'w', encoding='utf-8') as f:
        for key, wt in raw_edges.items():
            if key[0] != window:
                continue
            _, actor, concept = key
            record = {
                'actor':             actor,
                'concept':           concept,
                'window':            window,
                'weight':            wt,
                'weight_normalized': round(wt / n_arts, 4),
                'sources':           dict(source_breakdown[key]),
            }
            f.write(json.dumps(record, ensure_ascii=False) + '\n')
            n_written += 1
    print(f'Wrote {n_written:3d} edges to {out_path.name}')

print(f'\nDone. {len(windows_present)} edge files in {EDGES_DIR}')

## Step 7: SVO extraction on coref-resolved text (CAMEO-grounded verb filter)

Parallel exploratory track per README §6 — it **does not replace** the co-occurrence edges built in steps 4–6. SVO runs on the **coref-resolved** text from `03b_coref`, so pronoun subjects/objects ("it", "they", "he") surface as their actor antecedents — e.g. *it → strike → facility* becomes *Israel → strike → facility*.

To avoid a second parse pass, it **reuses** 03b's per-sentence `coref_resolved_text`: one resolved "body" is reconstructed per article by joining those sentences (with a character-offset table so every triple maps back to its exact `sentence_id`), and the single `en_core_web_trf` dependency-parse pass runs over those reconstructed bodies. No extra coreference or NER pass is added to the pipeline.

Each (subject, verb, object) triple is then filtered through `src/verb_map.py`: the lexical (main) verb is lemmatised; triples whose lemma is in `STOP_VERBS` or absent from `VERB_MAP` are dropped; survivors are tagged with their CAMEO quadrant and written to `data/output/edges/svo_triples.jsonl`. This is the slow cell (a full transformer pass); steps 4–6 are independent and unaffected.

In [ ]:
import time
import spacy
from textacy.extract import subject_verb_object_triples
from src.verb_map import VERB_MAP, STOP_VERBS

SVO_OUT = EDGES_DIR / 'svo_triples.jsonl'

# SVO runs on the COREF-RESOLVED text (03b), not the raw bodies, so pronoun
# subjects/objects ("it", "they") surface as their actor antecedents. To avoid a
# SECOND parse pass, reuse 03b's per-sentence coref_resolved_text: reconstruct one
# resolved "body" per article from those sentences (cheap string join) and run the
# single dependency-parse pass over those. No extra coref/NER pass is added.
coref_file = INTERIM_DIR / 'sentences_coref.jsonl'
assert coref_file.exists(), f'ERROR: {coref_file} not found — run 03b_coref first'
with open(coref_file, encoding='utf-8') as f:
    coref_sents = [json.loads(line) for line in f]

# Group sentences per article (file order) and reconstruct a resolved body, while
# recording each sentence's char span so a triple maps back to its exact
# sentence_id regardless of how spaCy re-segments the joined text.
by_article = defaultdict(list)
for s in coref_sents:
    by_article[s['article_id']].append(s)

bodies = []   # (article_id, resolved_body, [(start, end, sentence_id), ...], {sid: record})
for aid, sents in by_article.items():
    parts, spans, pos = [], [], 0
    for s in sents:
        t = s.get('coref_resolved_text', s['text'])
        spans.append((pos, pos + len(t), s['sentence_id']))
        parts.append(t)
        pos += len(t) + 1                       # +1 for the joining space
    bodies.append((aid, ' '.join(parts), spans, {s['sentence_id']: s for s in sents}))

print(f'Parsing {len(bodies)} coref-resolved bodies with en_core_web_trf for SVO '
      '(single pass — minutes, not seconds)...')

nlp = spacy.load('en_core_web_trf')

def main_verb(verb_tokens):
    """Lexical verb of an SVO verb span — skip auxiliaries (will, be, ...)."""
    for tok in reversed(list(verb_tokens)):
        if tok.pos_ == 'VERB':
            return tok
    return list(verb_tokens)[-1]

def locate(char_idx, spans):
    """Map a character offset in the resolved body back to its sentence_id."""
    for start, end, sid in spans:
        if start <= char_idx < end:
            return sid
    return None

triples_out  = []
n_extracted  = 0
n_stop       = 0
n_out_of_map = 0
cat_counts   = Counter()
t0 = time.time()

docs = nlp.pipe([b[1] for b in bodies], batch_size=16)
for idx, (doc, (aid, _body, spans, rec_by_sid)) in enumerate(zip(docs, bodies)):
    if (idx + 1) % 100 == 0 or idx == 0:
        print(f'  article {idx + 1:4d}/{len(bodies)}  elapsed={time.time() - t0:.0f}s')
    for svo in subject_verb_object_triples(doc):
        n_extracted += 1
        vtok  = main_verb(svo.verb)
        lemma = vtok.lemma_.lower()
        if lemma in STOP_VERBS:
            n_stop += 1
            continue
        if lemma not in VERB_MAP:
            n_out_of_map += 1
            continue
        cameo = VERB_MAP[lemma]
        cat_counts[cameo] += 1
        sid = locate(vtok.idx, spans)
        rec = rec_by_sid.get(sid, {})
        triples_out.append({
            'subject':        ' '.join(t.text for t in svo.subject),
            'verb_lemma':     lemma,
            'verb_surface':   vtok.text,
            'object':         ' '.join(t.text for t in svo.object),
            'cameo_category': cameo,
            'sentence_id':    sid,
            'window':         rec.get('window'),
            'date':           rec.get('date'),
            'source':         rec.get('source'),
        })

print(f'\nSVO parse done in {time.time() - t0:.0f}s')

with open(SVO_OUT, 'w', encoding='utf-8') as f:
    for t in triples_out:
        f.write(json.dumps(t, ensure_ascii=False) + '\n')

# ---- logging ----
n_retained = len(triples_out)
print(f'\nTriples extracted (raw)       : {n_extracted}')
print(f'  dropped - stop verbs        : {n_stop}')
print(f'  dropped - out of VERB_MAP    : {n_out_of_map}')
print(f'  retained (CAMEO-mapped)     : {n_retained}  '
      f'({100 * n_retained / max(n_extracted, 1):.1f}% of raw)')

print('\nRetained triples by CAMEO category:')
for cat in ('material_conflict', 'verbal_conflict',
            'verbal_cooperation', 'material_cooperation'):
    c = cat_counts.get(cat, 0)
    print(f'  {cat:22s}  {c:5d}  ({100 * c / max(n_retained, 1):.1f}%)')

print('\nTop 20 (subject, verb_lemma, object) triples by frequency:')
triple_freq = Counter((t['subject'], t['verb_lemma'], t['object']) for t in triples_out)
for (s, v, o), c in triple_freq.most_common(20):
    print(f'  {c:4d}  {s}  --{v}-->  {o}')

print(f'\nWrote {n_retained} SVO triples to {SVO_OUT}')
print('NOTE: SVO runs on coref-resolved text and is a parallel, exploratory track —')
print('      it does NOT replace the co-occurrence edges built in steps 4-6.')